In [1]:
import datasets

ds = datasets.load_dataset("open-r1/DAPO-Math-17k-Processed")

/home/recoverx/astarag/trainer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 17398/17398 [00:00<00:00, 264995.12 examples/s]


In [6]:
type(ds)

datasets.dataset_dict.DatasetDict

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
        num_rows: 17398
    })
})

In [5]:
ds['train'][0]

{'prompt': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a

In [4]:
import os
os.chdir("/home/recoverx/astarag/trainer")

In [5]:
# Test DAPO processed environment
from trainer.envs.math_env import (
    DAPOMath17KProcessedDataset,
    DAPOMath17KProcessedEnv,
    extract_think_and_answer,
    DAPO_MATH_SYSTEM_PROMPT
)
from trainer.envs import base_env
import transformers

print("✓ Imports successful")

/home/recoverx/astarag/trainer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/recoverx/astarag/trainer/.venv/lib/python3.13/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✓ Imports successful


In [6]:
# Test 1: Test extract_think_and_answer function
print("=" * 60)
print("Test 1: Testing extract_think_and_answer function")
print("=" * 60)

# Test case 1: Both tags present
test_text1 = """
<think>
This is the reasoning process.
Step 1: Calculate 2 + 2
Step 2: The answer is 4
</think>
<answer>4</answer>
"""
think1, answer1 = extract_think_and_answer(test_text1)
print(f"Test 1.1 - Both tags present:")
print(f"  Think text: {think1}")
print(f"  Answer text: {answer1}")
assert think1 == "This is the reasoning process.\nStep 1: Calculate 2 + 2\nStep 2: The answer is 4", "Think text mismatch"
assert answer1 == "4", "Answer text mismatch"
print("  ✓ PASSED\n")

# Test case 2: Missing think tag
test_text2 = "<answer>5</answer>"
think2, answer2 = extract_think_and_answer(test_text2)
print(f"Test 1.2 - Missing think tag:")
print(f"  Think text: {think2}")
print(f"  Answer text: {answer2}")
assert think2 is None, "Think should be None"
assert answer2 == "5", "Answer text mismatch"
print("  ✓ PASSED\n")

# Test case 3: Missing answer tag
test_text3 = "<think>Some reasoning</think>"
think3, answer3 = extract_think_and_answer(test_text3)
print(f"Test 1.3 - Missing answer tag:")
print(f"  Think text: {think3}")
print(f"  Answer text: {answer3}")
assert think3 == "Some reasoning", "Think text mismatch"
assert answer3 is None, "Answer should be None"
print("  ✓ PASSED\n")

# Test case 4: Both missing
test_text4 = "No tags here"
think4, answer4 = extract_think_and_answer(test_text4)
print(f"Test 1.4 - Both tags missing:")
print(f"  Think text: {think4}")
print(f"  Answer text: {answer4}")
assert think4 is None, "Think should be None"
assert answer4 is None, "Answer should be None"
print("  ✓ PASSED\n")

print("All extract_think_and_answer tests passed! ✓")

Test 1: Testing extract_think_and_answer function
Test 1.1 - Both tags present:
  Think text: This is the reasoning process.
Step 1: Calculate 2 + 2
Step 2: The answer is 4
  Answer text: 4
  ✓ PASSED

Test 1.2 - Missing think tag:
  Think text: None
  Answer text: 5
  ✓ PASSED

Test 1.3 - Missing answer tag:
  Think text: Some reasoning
  Answer text: None
  ✓ PASSED

Test 1.4 - Both tags missing:
  Think text: None
  Answer text: None
  ✓ PASSED

All extract_think_and_answer tests passed! ✓


In [7]:
# Test 2: Test DAPOMath17KProcessedDataset.prepare_dataset
print("=" * 60)
print("Test 2: Testing DAPOMath17KProcessedDataset.prepare_dataset")
print("=" * 60)

dataset_processor = DAPOMath17KProcessedDataset()
ds_name = "open-r1/DAPO-Math-17k-Processed"

print(f"Loading and processing dataset: {ds_name}")
print("This may take a moment due to parallel processing...")

prompts = dataset_processor.prepare_dataset(ds_name)

print(f"\n✓ Dataset loaded successfully!")
print(f"  Total prompts: {len(prompts)}")
print(f"  Type: {type(prompts)}")
print(f"  Expected type: list")

assert isinstance(prompts, list), "prompts should be a list"
assert len(prompts) > 0, "prompts list should not be empty"
print("  ✓ PASSED\n")

# Check structure of first prompt
print("Checking structure of first prompt:")
first_prompt = prompts[0]
print(f"  Type: {type(first_prompt)}")
print(f"  Expected type: {base_env.Prompt}")
assert isinstance(first_prompt, base_env.Prompt), "First item should be a Prompt object"
print("  ✓ PASSED\n")

# Check prompt fields
print("Checking prompt fields:")
print(f"  prompt: {type(first_prompt.prompt)} (length: {len(first_prompt.prompt) if isinstance(first_prompt.prompt, list) else 'N/A'})")
print(f"  data_source: {type(first_prompt.data_source)}")
print(f"  ability: {first_prompt.ability}")
print(f"  reward_model: {type(first_prompt.reward_model)}")
print(f"  extra_info: {type(first_prompt.extra_info)}")

assert isinstance(first_prompt.prompt, list), "prompt should be a list"
assert len(first_prompt.prompt) == 2, "prompt should have 2 messages (system + user)"
assert first_prompt.prompt[0]["role"] == "system", "First message should be system"
assert first_prompt.prompt[1]["role"] == "user", "Second message should be user"
assert first_prompt.ability == "math", "ability should be 'math'"
print("  ✓ PASSED\n")

# Check system prompt
print("Checking system prompt:")
system_content = first_prompt.prompt[0]["content"]
print(f"  System prompt matches expected: {system_content == DAPO_MATH_SYSTEM_PROMPT}")
assert system_content == DAPO_MATH_SYSTEM_PROMPT, "System prompt should match DAPO_MATH_SYSTEM_PROMPT"
print("  ✓ PASSED\n")

# Display sample prompt structure
print("Sample prompt structure:")
print(f"  System message: {first_prompt.prompt[0]['content'][:100]}...")
print(f"  User message: {first_prompt.prompt[1]['content'][:100]}...")
print(f"  Reward model keys: {list(first_prompt.reward_model.keys()) if isinstance(first_prompt.reward_model, dict) else 'N/A'}")

print("\nAll dataset preparation tests passed! ✓")

Test 2: Testing DAPOMath17KProcessedDataset.prepare_dataset
Loading and processing dataset: open-r1/DAPO-Math-17k-Processed
This may take a moment due to parallel processing...


Map: 100%|██████████| 17398/17398 [00:01<00:00, 14223.61 examples/s]



✓ Dataset loaded successfully!
  Total prompts: 17398
  Type: <class 'list'>
  Expected type: list
  ✓ PASSED

Checking structure of first prompt:
  Type: <class 'trainer.envs.base_env.Prompt'>
  Expected type: <class 'trainer.envs.base_env.Prompt'>
  ✓ PASSED

Checking prompt fields:
  prompt: <class 'list'> (length: 2)
  data_source: <class 'str'>
  ability: math
  reward_model: <class 'dict'>
  extra_info: <class 'dict'>
  ✓ PASSED

Checking system prompt:
  System prompt matches expected: True
  ✓ PASSED

Sample prompt structure:
  System message: You are a helpful math assistant. 

For every response, please provide a step-by-step reasoning proc...
  User message: In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outsi...
  Reward model keys: ['ground_truth', 'style']

All dataset preparation tests passed! ✓


In [8]:
# Test 3: Test DAPOMath17KProcessedEnv
print("=" * 60)
print("Test 3: Testing DAPOMath17KProcessedEnv")
print("=" * 60)

# Use the first prompt from the dataset
test_prompt = prompts[0]
print(f"Using test prompt from dataset")
print(f"  Prompt ability: {test_prompt.ability}")
print(f"  Reward model: {test_prompt.reward_model}")

# Create environment instance
env = DAPOMath17KProcessedEnv(prompt=test_prompt, tokenizer=None)
print("  ✓ Environment created successfully\n")

# Test reset
print("Test 3.1: Testing reset()")
obs, info = env.reset()
print(f"  Observation type: {type(obs)}")
print(f"  Info: {info}")
assert isinstance(obs, base_env.Prompt), "reset should return a Prompt object"
assert obs == test_prompt, "reset should return the same prompt"
print("  ✓ PASSED\n")

# Test step with correct answer (both think and answer present)
print("Test 3.2: Testing step() with correct answer")
gt = test_prompt.reward_model.get("ground_truth", "test_answer")
correct_action = f"""
<think>
Let me think about this step by step.
The answer should be {gt}.
</think>
<answer>{gt}</answer>
"""
result = env.step(correct_action, meta_info={"logps": [0.5, 0.3]})
print(f"  Reward: {result.reward}")
print(f"  Terminated: {result.terminated}")
print(f"  Done: {result.done}")
print(f"  Info: {result.info}")
assert result.reward == 2.0, f"Expected reward 2.0, got {result.reward}"
assert result.terminated == True, "Should be terminated"
assert result.done == True, "Should be done"
assert result.info.get("inference_engine_logps") == [0.5, 0.3], "Logps should be preserved"
print("  ✓ PASSED\n")

# Test step with missing think tag
print("Test 3.3: Testing step() with missing think tag")
action_no_think = f"<answer>{gt}</answer>"
result2 = env.step(action_no_think, meta_info=None)
print(f"  Reward: {result2.reward}")
assert result2.reward == -1.0, f"Expected reward -1.0 (missing think), got {result2.reward}"
print("  ✓ PASSED\n")

# Test step with missing answer tag
print("Test 3.4: Testing step() with missing answer tag")
action_no_answer = "<think>Some reasoning</think>"
result3 = env.step(action_no_answer, meta_info=None)
print(f"  Reward: {result3.reward}")
assert result3.reward == -1.0, f"Expected reward -1.0 (missing answer), got {result3.reward}"
print("  ✓ PASSED\n")

# Test step with wrong answer
print("Test 3.5: Testing step() with wrong answer")
wrong_action = """
<think>Some reasoning</think>
<answer>wrong_answer</answer>
"""
result4 = env.step(wrong_action, meta_info=None)
print(f"  Reward: {result4.reward}")
assert result4.reward == 0.0, f"Expected reward 0.0 (wrong answer), got {result4.reward}"
print("  ✓ PASSED\n")

# Test step with both tags missing
print("Test 3.6: Testing step() with both tags missing")
no_tags_action = "Just some text without tags"
result5 = env.step(no_tags_action, meta_info=None)
print(f"  Reward: {result5.reward}")
assert result5.reward == -2.0, f"Expected reward -2.0 (both missing), got {result5.reward}"
print("  ✓ PASSED\n")

print("All environment tests passed! ✓")

Test 3: Testing DAPOMath17KProcessedEnv
Using test prompt from dataset
  Prompt ability: math
  Reward model: {'ground_truth': '34', 'style': 'rule-lighteval/MATH_v2'}
  ✓ Environment created successfully

Test 3.1: Testing reset()
  Observation type: <class 'trainer.envs.base_env.Prompt'>
  Info: {}
  ✓ PASSED

Test 3.2: Testing step() with correct answer
  Reward: 2.0
  Terminated: True
  Done: True
  Info: {'inference_engine_logps': [0.5, 0.3]}
  ✓ PASSED

Test 3.3: Testing step() with missing think tag
  Reward: -1.0
  ✓ PASSED

Test 3.4: Testing step() with missing answer tag
  Reward: -1.0
  ✓ PASSED

Test 3.5: Testing step() with wrong answer
  Reward: 0.0
  ✓ PASSED

Test 3.6: Testing step() with both tags missing
  Reward: -2.0
  ✓ PASSED

All environment tests passed! ✓


In [9]:
# Test 4: Verify parallel processing worked correctly
print("=" * 60)
print("Test 4: Verifying parallel processing")
print("=" * 60)

# Check that all prompts have the correct structure
print("Checking consistency across multiple prompts...")
sample_size = min(100, len(prompts))
inconsistent = 0

for i in range(sample_size):
    p = prompts[i]
    if not isinstance(p, base_env.Prompt):
        inconsistent += 1
        continue
    if not isinstance(p.prompt, list) or len(p.prompt) != 2:
        inconsistent += 1
        continue
    if p.prompt[0]["role"] != "system" or p.prompt[1]["role"] != "user":
        inconsistent += 1
        continue
    if p.ability != "math":
        inconsistent += 1
        continue

print(f"  Checked {sample_size} prompts")
print(f"  Inconsistent prompts: {inconsistent}")
assert inconsistent == 0, f"Found {inconsistent} inconsistent prompts"
print("  ✓ All checked prompts have consistent structure\n")

# Verify system prompt is consistent
print("Verifying system prompt consistency...")
system_prompts = [p.prompt[0]["content"] for p in prompts[:sample_size]]
unique_system_prompts = set(system_prompts)
print(f"  Unique system prompts: {len(unique_system_prompts)}")
assert len(unique_system_prompts) == 1, "All system prompts should be identical"
assert list(unique_system_prompts)[0] == DAPO_MATH_SYSTEM_PROMPT, "System prompt should match constant"
print("  ✓ PASSED\n")

print("All parallel processing verification tests passed! ✓")

Test 4: Verifying parallel processing
Checking consistency across multiple prompts...
  Checked 100 prompts
  Inconsistent prompts: 0
  ✓ All checked prompts have consistent structure

Verifying system prompt consistency...
  Unique system prompts: 1
  ✓ PASSED

All parallel processing verification tests passed! ✓


In [10]:
# Test 5: Summary and final verification
print("=" * 60)
print("Test 5: Final Summary")
print("=" * 60)

print(f"✓ Dataset loaded: {len(prompts)} prompts")
print(f"✓ All prompts are base_env.Prompt instances")
print(f"✓ All prompts have correct structure (system + user messages)")
print(f"✓ System prompt is consistent across all prompts")
print(f"✓ extract_think_and_answer function works correctly")
print(f"✓ DAPOMath17KProcessedEnv works correctly with all test cases")
print(f"✓ Parallel processing completed successfully")

print("\n" + "=" * 60)
print("🎉 ALL TESTS PASSED! Everything is working as expected.")
print("=" * 60)

Test 5: Final Summary
✓ Dataset loaded: 17398 prompts
✓ All prompts are base_env.Prompt instances
✓ All prompts have correct structure (system + user messages)
✓ System prompt is consistent across all prompts
✓ extract_think_and_answer function works correctly
✓ DAPOMath17KProcessedEnv works correctly with all test cases
✓ Parallel processing completed successfully

🎉 ALL TESTS PASSED! Everything is working as expected.
